# TaaS Core Studio — Colab T4 deployment

**Instruction guide:** follow [the step-by-step operator guide](../../docs/deployment/COLAB_T4_QUICKSTART.md) before running this notebook. It covers runtime selection, Secrets, ModelProfile, launch, browser access, troubleshooting and shutdown.

This notebook runs the TaaS Studio web application **inside the Colab VM**. Select a T4 GPU runtime when available. The current milestone provides GoalSpec/tool metadata input and TaaS discovery results; MCP invocation, agent orchestration, live execution streaming and durable run history are not implemented.

**Security:** owner access uses Colab's notebook port proxy. Public team access is disabled unless you deliberately configure a named Cloudflare Tunnel plus a Cloudflare Access application/policy. A tunnel URL alone is not authentication.

In [ ]:
import os, sys, subprocess, pathlib, json, time, shutil, urllib.request
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "3f74c753a9a93629ecee2718620fb06bd3affc82"  # pinned UI/API revision
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
APP_HOST, APP_PORT = "127.0.0.1", 8000

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
print("Pinned TaaS commit:", actual)

## Step 2 — Confirm the GPU

Run the next cell and confirm that Colab allocated an NVIDIA Tesla T4. If no GPU is visible, select **Runtime → Change runtime type → T4 GPU**, reconnect, and rerun from Step 1. Studio's discovery API can run on CPU, but local GPU-backed inference cannot.

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
if GPU_AVAILABLE:
    print(gpu_check.stdout)
else:
    print("No NVIDIA GPU visible; continuing CPU-only.")
print("Python:", sys.version.split()[0])
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)"))

## Step 3 — Install TaaS Studio

Run this cell after the repository checkout. It installs the optional FastAPI/Uvicorn dependencies and checks the application import. Expected output: `Studio imports OK`.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[studio]"], cwd=WORKDIR, check=True)
subprocess.run([sys.executable, "-c", "import fastapi, uvicorn, taas.studio.api; print('Studio imports OK')"], cwd=WORKDIR, check=True)

## Step 4 — Configure Secrets and ModelProfile

Before running the next cell:

1. Open Colab's **Secrets** panel (key icon).
2. Add `TAAS_STUDIO_API_TOKEN` with a strong random value (at least 24 characters).
3. Grant this notebook access to the secret.
4. Prepare a valid TaaS ModelProfile JSON using `src/taas/model_profile.py` and repository examples as the schema authority.
5. Upload exactly one reviewed JSON file when prompted.

Do not put provider API keys in the JSON or print secrets. If you do not have a valid ModelProfile, stop and prepare one; do not guess its schema.

In [ ]:
from google.colab import userdata, files

api_token = userdata.get("TAAS_STUDIO_API_TOKEN")
if not api_token or len(api_token) < 24:
    raise RuntimeError("Set TAAS_STUDIO_API_TOKEN in Colab Secrets (minimum 24 characters).")
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token

uploaded = files.upload()
json_files = [name for name in uploaded if name.lower().endswith(".json")]
if len(json_files) != 1:
    raise RuntimeError("Upload exactly one reviewed ModelProfile JSON file.")
profile_path = pathlib.Path("/content") / json_files[0]
profile_obj = json.loads(profile_path.read_text(encoding="utf-8"))
if not isinstance(profile_obj, dict):
    raise ValueError("ModelProfile must be a JSON object.")
os.environ["TAAS_MODEL_PROFILE"] = str(profile_path.resolve())
print("TaaS API credential and ModelProfile configured.")

## Step 5 — Optional local model endpoint

The current TaaS discovery engine is CPU-oriented. A T4 helps only when a CUDA-capable model server (LLM, embeddings or reranker) is installed, running and actually used by a configured provider adapter. This notebook does not download or launch a model automatically.

For now, leave the endpoint unset unless you have independently deployed and tested a compatible OpenAI-style model server. A T4 is memory-constrained; use a quantized model that fits available VRAM.

In [ ]:
LOCAL_MODEL_BASE_URL = os.environ.get("TAAS_LOCAL_MODEL_BASE_URL", "").strip()
if LOCAL_MODEL_BASE_URL:
    print("Local model endpoint configured:", LOCAL_MODEL_BASE_URL)
else:
    print("No local model endpoint; the configured ModelProfile/provider will be used.")

## Step 6 — Launch TaaS Studio on the Colab VM

Run the next cell. It starts Uvicorn on loopback (`127.0.0.1:8000`), waits for the health endpoint and confirms that the browser UI responds. Keep the runtime connected while using Studio. If startup fails, inspect `/content/taas-studio.log`.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    studio_process.wait(timeout=10)

studio_log = open("/content/taas-studio.log", "w")
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=30):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HOST}:{APP_PORT}/health"))
print("Browser UI HTTP status:", wait_http(f"http://{APP_HOST}:{APP_PORT}/")[0])

## Step 7 — Open Studio in your browser

Run the next cell and click **Open TaaS Core Studio**. This Colab-proxied link is associated with the active notebook/runtime session; it is not a permanent URL. If it fails, confirm the runtime is connected and rerun this cell.

In [ ]:
from google.colab import output

proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Core Studio</a></p>'))
print("Colab session-scoped URL:", proxy_url)

## Optional — Protected team access

Skip this for personal use. Team sharing requires a named Cloudflare Tunnel and a Cloudflare Access application/policy configured **before** enabling the next cell:

- Route your chosen hostname to `http://127.0.0.1:8000`.
- Protect the entire hostname with an allow policy for intended identities.
- Store the tunnel token in Colab Secrets as `CF_TUNNEL_TOKEN`.
- Ensure the Cloudflare tunnel client (`cloudflared`) is installed and version-pinned.
- Then set `enable_cf = True`.

A tunnel token provides connectivity, not user authentication. Do not use an unauthenticated quick tunnel.

In [ ]:
enable_cf = False  # Enable only after named tunnel + Cloudflare Access policy are configured.
if enable_cf:
    cf_token = userdata.get("CF_TUNNEL_TOKEN")
    if not cf_token:
        raise RuntimeError("Add CF_TUNNEL_TOKEN to Colab Secrets.")
    cloudflared = shutil.which("cloudflared")
    if not cloudflared:
        raise RuntimeError("Install and pin cloudflared first; do not download an unpinned binary.")
    cf_log = open("/content/cloudflared.log", "w")
    cf_process = subprocess.Popen(
        [cloudflared, "tunnel", "run", "--token", cf_token],
        stdout=cf_log, stderr=subprocess.STDOUT
    )
    print("Tunnel process started. Verify the configured Cloudflare Access hostname before sharing.")
else:
    print("Public tunnel disabled; use the Colab session proxy.")

## Step 8 — Stop Studio

Run the final cell when finished. Colab may also terminate the runtime automatically. Files in `/content` and model weights in RAM/VRAM are temporary; do not treat them as durable experiment storage.

In [ ]:
if "cf_process" in globals() and cf_process.poll() is None:
    cf_process.terminate()
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    studio_process.wait(timeout=10)
studio_log.close()
print("TaaS Studio stopped.")